In [1]:
!pip install transformers>=4.35.0
!pip install datasets>=2.14.0
!pip install evaluate>=0.4.0
!pip install rouge-score>=0.1.2
!pip install streamlit>=1.28.0
!pip install pandas numpy matplotlib seaborn
!pip install sentencepiece protobuf accelerate
!pip install scikit-learn
!pip install evaluate
!pip install -q rouge_score

In [2]:
from datasets import load_dataset
from transformers import T5Tokenizer

ds = load_dataset("b-mc2/wikihow_lists")


ds = ds["train"].select_columns(["title", "result"]).shuffle(seed=42)

total = ds.shape[0]
train_size = int(total * 0.8)
test_size = int(total * 0.1)
val_size = total - train_size - test_size

train_ds = ds.select(range(0, train_size))
test_ds = ds.select(range(train_size, train_size + test_size))
val_ds = ds.select(range(train_size + test_size, total))

SYSTEM_PROMPT = (
    "You are a concise text summarizer. Read the step-by-step instructions "
    "and output ONLY a short, imperative text that summarizes."
)

def format_for_sft(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Instructions:\n{example['result']}"},
            {"role": "assistant", "content": example["title"]},
        ]
    }

train_dataset = train_ds.map(format_for_sft,remove_columns=["title", "result"])
val_dataset = val_ds.map(format_for_sft, remove_columns=["title", "result"])
print(len(val_dataset))


README.md:   0%|          | 0.00/775 [00:00<?, ?B/s]

wikihow_lists.csv:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11461 [00:00<?, ? examples/s]

Map:   0%|          | 0/9168 [00:00<?, ? examples/s]

Map:   0%|          | 0/1147 [00:00<?, ? examples/s]

1147


In [3]:
!pip install -U trl peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 50.3 MB/s  0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.21.2
    Uninstalling peft-0.21.2:
      Successfully uninstalled peft-0.21.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [peft]


In [4]:
!pip install -U bitsandbytes>=0.46.1

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from collections import Counter

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 1. Use `dtype=` (not `torch_dtype=`)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

# 2. Prepare k-bit model & apply LoRA manually so all dtypes are strictly float16/float32
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)
model = get_peft_model(model, lora_config)

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

training_args = SFTConfig(
    output_dir="./qwen_wikihow_lora",
    max_length=512,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=2,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_steps=100,
    fp16=True,
    bf16=False,
    assistant_only_loss=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer
)

for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

print(Counter(p.dtype for p in trainer.model.parameters() if p.requires_grad))  # should be only float32

trainer.train()

trainer.model.save_pretrained("qwen")
tokenizer.save_pretrained("qwen")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

... (10 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/1147 [00:00<?, ? examples/s]

Counter({torch.float32: 392})


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
500,1.375621,1.308073,1.337890,0.727559,846193.000000
1000,1.016382,1.322762,1.077356,0.725727,1695129.000000
1146,1.018771,1.322327,1.067593,0.727623,1941186.000000


('qwen/tokenizer_config.json',
 'qwen/chat_template.jinja',
 'qwen/tokenizer.json')

In [12]:
from evaluate import load
import time
import json
import pandas as pd
from tqdm import tqdm

SYSTEM_PROMPT = (
     "You are a concise text summarizer. Read the step-by-step instructions "
    "and output ONLY a short, imperative text that summarizes."
)

def build_prompt(steps_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Instructions:\n{steps_text}"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

rouge = load("rouge")

def evaluate_finetuned_llm(model, tokenizer, test_ds, batch_size=16):
    predictions = []
    references = []
    start_time = time.perf_counter()

    for i in tqdm(range(0, len(test_ds), batch_size)):
        batch = test_ds[i : i + batch_size]
        prompts = [build_prompt(doc) for doc in batch["result"]]

        inputs = tokenizer(
            prompts,
            max_length=512,
            truncation=True,
            padding=True,
            padding_side='left',
            return_tensors="pt"
        ).to(model.device)

        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=24,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # Decode only the newly generated assistant response
        generated_tokens = outputs[:, input_len:]
        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        cleaned_preds = [pred.strip().strip('"').strip("'") for pred in decoded_preds]

        predictions.extend(cleaned_preds)
        references.extend(batch["title"])

    total_time = time.perf_counter() - start_time

    # Compute ROUGE identical to your T5 evaluation
    scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    clean_scores = {k: round(float(v), 4) for k, v in scores.items()}
    clean_scores["total_time_sec"] = round(total_time, 2)
    clean_scores["latency_ms_per_sample"] = round((total_time / len(test_ds)) * 1000, 2)

    # Save aggregate metrics
    with open("qwen_finetuned_rouge.json", "w") as f:
        json.dump(clean_scores, f, indent=4)

    # Save sample-level predictions for side-by-side comparison with T5
    pd.DataFrame({
        "input_steps": test_ds["result"],
        "reference_title": references,
        "qwen_predicted_title": predictions
    }).to_csv("qwen_test_predictions.csv", index=False)

    return clean_scores

In [8]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 80.5 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [14]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.float16
)


model = PeftModel.from_pretrained(
    base_model,
    "/content/qwen_wikihow_lora/checkpoint-500"
)

results = evaluate_finetuned_llm(model, tokenizer, test_ds, batch_size=16)
print("Fine-tuned LLM Results:", results)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

100%|██████████| 72/72 [02:05<00:00,  1.75s/it]


Fine-tuned LLM Results: {'rouge1': 0.451, 'rouge2': 0.2133, 'rougeL': 0.4406, 'rougeLsum': 0.4409, 'total_time_sec': 125.7, 'latency_ms_per_sample': 109.69}


In [15]:
!zip -r qwen.zip /content/qwen_wikihow_lora/checkpoint-500

  adding: content/qwen_wikihow_lora/checkpoint-500/ (stored 0%)
  adding: content/qwen_wikihow_lora/checkpoint-500/scheduler.pt (deflated 61%)
  adding: content/qwen_wikihow_lora/checkpoint-500/scaler.pt (deflated 64%)
  adding: content/qwen_wikihow_lora/checkpoint-500/README.md (deflated 65%)
  adding: content/qwen_wikihow_lora/checkpoint-500/rng_state.pth (deflated 26%)
  adding: content/qwen_wikihow_lora/checkpoint-500/adapter_model.safetensors (deflated 8%)
  adding: content/qwen_wikihow_lora/checkpoint-500/training_args.bin (deflated 53%)
  adding: content/qwen_wikihow_lora/checkpoint-500/tokenizer.json (deflated 81%)
  adding: content/qwen_wikihow_lora/checkpoint-500/adapter_config.json (deflated 61%)
  adding: content/qwen_wikihow_lora/checkpoint-500/tokenizer_config.json (deflated 59%)
  adding: content/qwen_wikihow_lora/checkpoint-500/chat_template.jinja (deflated 71%)
  adding: content/qwen_wikihow_lora/checkpoint-500/trainer_state.json (deflated 65%)
  adding: content/qwen_w